In [1]:
from pathlib import Path
import pandas as pd

# Paths
project_root = Path(".").resolve()
csv_path = project_root / "data/processed/icu_stay_modeling_24h_v1_1.csv"
out_path = project_root / "outputs/tables/icu_stay_modeling_24h_v1_1_schema.csv"

# Load data
df = pd.read_csv(csv_path)

# Basic role rules
target_cols = {"prolonged_los_8d"}
audit_cols = {
    "subject_id", "hadm_id", "stay_id",
    "intime", "outtime", "los_days",
    "first_careunit", "last_careunit", "admission_type"
}

def guess_role(col: str) -> str:
    if col in target_cols:
        return "target"
    if col in audit_cols:
        return "audit_only"
    return "predictor"

def guess_feature_family(col: str) -> str:
    if col in target_cols:
        return "target"
    if col in audit_cols:
        return "audit"
    if col.endswith("_grp"):
        return "grouped_context"
    if col.startswith("has_"):
        return "presence_flag"
    if "chartevents" in col or "missing_core_vitals" in col:
        return "measurement_density_or_missingness"
    if any(col.startswith(prefix) for prefix in ["hr_", "rr_", "spo2_", "temp_", "map_"]):
        return "vitals_summary"
    if any(col.startswith(prefix) for prefix in ["creatinine_", "wbc_", "hemoglobin_", "lactate_"]):
        return "lab_summary"
    return "other"

rows = []
for col in df.columns:
    series = df[col]
    non_null = int(series.notna().sum())
    nulls = int(series.isna().sum())
    pct_null = round(nulls / len(df), 4) if len(df) else 0.0
    nunique = int(series.nunique(dropna=True))

    example_vals = series.dropna().astype(str).head(3).tolist()
    example_value = " | ".join(example_vals) if example_vals else ""

    rows.append({
        "column_name": col,
        "pandas_dtype": str(series.dtype),
        "role": guess_role(col),
        "feature_family": guess_feature_family(col),
        "non_null_count": non_null,
        "null_count": nulls,
        "pct_null": pct_null,
        "n_unique_non_null": nunique,
        "example_value": example_value,
    })

schema_df = pd.DataFrame(rows)

# Nice ordering
role_order = {"target": 0, "audit_only": 1, "predictor": 2}
schema_df["role_sort"] = schema_df["role"].map(role_order).fillna(99)
schema_df = schema_df.sort_values(["role_sort", "feature_family", "column_name"]).drop(columns="role_sort")

out_path.parent.mkdir(parents=True, exist_ok=True)
schema_df.to_csv(out_path, index=False)

print(f"Created: {out_path}")
print(schema_df.head(15).to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: '/home/luis/Documents/mdms_pitt/courses/Capstone/projects/case-studies-capstone/notebooks/data/processed/icu_stay_modeling_24h_v1_1.csv'

In [3]:

import pandas as pd
df = pd.read_csv("data/processed/icu_stay_modeling_24h_v1_1.csv")
print("Rows:", len(df))
print("Distinct stay_id:", df["stay_id"].nunique())
print("Columns:", len(df.columns))
print("\nTarget counts:")
print(df["prolonged_los_8d"].value_counts(dropna=False).sort_index())
print("\nTarget prevalence:")
print(df["prolonged_los_8d"].value_counts(normalize=True).sort_index().round(4))


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/icu_stay_modeling_24h_v1_1.csv'